# Milestone 1: Quaternion Math Sanity Checks

This notebook stress-tests the quaternion algebra implementation before any ML experiments. The goal is to catch Hamilton-product or shape-convention mistakes early, because even a small sign or ordering bug would invalidate later comparisons.

In [1]:
from pathlib import Path
import sys

import torch

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
repo_root = next(path for path in candidates if (path / "src" / "qvnn_pc").exists())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from qvnn_pc.quaternion import hamilton_product, q_conj, q_matmul, q_mul, q_norm

torch.set_printoptions(precision=4, sci_mode=True)
torch.manual_seed(0)


In [2]:
def check(name, fn):
    fn()
    print(f"Passed: {name}")


## Basis Rules

These are the quickest way to catch sign or ordering mistakes in the Hamilton product.

In [3]:
one = torch.tensor([1.0, 0.0, 0.0, 0.0])
minus_one = torch.tensor([-1.0, 0.0, 0.0, 0.0])
i = torch.tensor([0.0, 1.0, 0.0, 0.0])
j = torch.tensor([0.0, 0.0, 1.0, 0.0])
k = torch.tensor([0.0, 0.0, 0.0, 1.0])

def check_basis_rules():
    torch.testing.assert_close(q_mul(i, i), minus_one)
    torch.testing.assert_close(q_mul(j, j), minus_one)
    torch.testing.assert_close(q_mul(k, k), minus_one)
    torch.testing.assert_close(q_mul(i, j), k)
    torch.testing.assert_close(q_mul(j, k), i)
    torch.testing.assert_close(q_mul(k, i), j)
    torch.testing.assert_close(q_mul(j, i), -k)
    torch.testing.assert_close(q_mul(k, j), -i)
    torch.testing.assert_close(q_mul(i, k), -j)

check("basis multiplication rules", check_basis_rules)


Passed: basis multiplication rules


## Identity, Conjugate, and Norm

These are the milestone's core invariants, plus a couple of extra consistency checks.

In [4]:
q = torch.tensor([1.5, -2.0, 0.25, 3.0])
q_alt = torch.tensor([-0.75, 1.25, 2.5, -1.0])

def check_identity():
    torch.testing.assert_close(q_mul(q, one), q)
    torch.testing.assert_close(q_mul(one, q), q)

def check_conjugate_and_norm():
    conjugate_product = q_mul(q, q_conj(q))
    reverse_conjugate_product = q_mul(q_conj(q), q)
    norm_squared = q_norm(q).square()
    expected = torch.stack((norm_squared, q.new_tensor(0.0), q.new_tensor(0.0), q.new_tensor(0.0)))

    torch.testing.assert_close(conjugate_product, expected)
    torch.testing.assert_close(reverse_conjugate_product, expected)
    torch.testing.assert_close(q_conj(q_conj(q)), q)
    torch.testing.assert_close(q_norm(q).square(), q.square().sum())

check("q * 1 = q and 1 * q = q", check_identity)
check("q * conj(q) = conj(q) * q = norm(q)^2", check_conjugate_and_norm)


Passed: q * 1 = q and 1 * q = q
Passed: q * conj(q) = conj(q) * q = norm(q)^2


In [5]:
q1 = torch.tensor([1.0, 2.0, -1.0, 0.5])
q2 = torch.tensor([0.5, -1.0, 3.0, 2.0])

def check_non_commutativity():
    left_then_right = q_mul(q1, q2)
    right_then_left = q_mul(q2, q1)
    assert not torch.allclose(left_then_right, right_then_left), (
        "Hamilton product should be non-commutative for generic quaternions."
    )
    assert not torch.allclose(q_mul(q, q_alt), q_mul(q_alt, q))

check("q1 * q2 != q2 * q1 for generic q1, q2", check_non_commutativity)


Passed: q1 * q2 != q2 * q1 for generic q1, q2


## Quaternion Matmul

The shape checks are required for the milestone. We also compare a tiny example against an explicit manual sum to verify the left-multiplication convention.

In [6]:
def check_q_matmul_manual_example():
    W = torch.tensor(
        [
            [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]],
            [[0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]],
        ]
    )
    x = torch.tensor(
        [
            [2.0, -1.0, 0.5, 3.0],
            [0.25, 2.0, -0.5, 1.0],
        ]
    )

    expected = torch.stack(
        [
            q_mul(W[0, 0], x[0]) + q_mul(W[0, 1], x[1]),
            q_mul(W[1, 0], x[0]) + q_mul(W[1, 1], x[1]),
        ]
    )
    torch.testing.assert_close(q_matmul(W, x), expected)

def check_q_matmul_shapes():
    W = torch.randn(3, 2, 4)
    x = torch.randn(5, 2, 4)
    y = q_matmul(W, x)
    assert y.shape == (5, 3, 4)

    W_batched = torch.randn(4, 3, 2, 4)
    x_batched = torch.randn(4, 2, 4)
    y_batched = q_matmul(W_batched, x_batched)
    assert y_batched.shape == (4, 3, 4)

    W_extra_batch = torch.randn(2, 4, 3, 2, 4)
    x_extra_batch = torch.randn(2, 4, 2, 4)
    y_extra_batch = q_matmul(W_extra_batch, x_extra_batch)
    assert y_extra_batch.shape == (2, 4, 3, 4)

def check_q_matmul_rejects_bad_shapes():
    bad_W = torch.randn(3, 2, 3)
    good_x = torch.randn(2, 4)
    try:
        q_matmul(bad_W, good_x)
        raise AssertionError("Expected q_matmul to reject tensors whose last dimension is not 4.")
    except ValueError:
        pass

    W = torch.randn(3, 2, 4)
    x = torch.randn(5, 5, 4)
    try:
        q_matmul(W, x)
        raise AssertionError("Expected q_matmul to reject incompatible input feature counts.")
    except ValueError:
        pass

check("q_matmul manual left-multiplication example", check_q_matmul_manual_example)
check("q_matmul shape tests", check_q_matmul_shapes)
check("q_matmul rejects bad shapes", check_q_matmul_rejects_bad_shapes)


Passed: q_matmul manual left-multiplication example
Passed: q_matmul shape tests
Passed: q_matmul rejects bad shapes


In [7]:
print("All Milestone 1 quaternion sanity checks passed.")


All Milestone 1 quaternion sanity checks passed.
